In [17]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.applications.efficientnet import preprocess_input

print("EfficientNet preprocessing imported successfully.")

from tensorflow.keras import Model

sns.set_theme(style="whitegrid")



EfficientNet preprocessing imported successfully.


In [2]:
# Load the selected product dataset
df = pd.read_csv("fashion_products_selected_v2.csv")

print("Dataset shape:", df.shape)
display(
    df[
        [
            "id",
            "productDisplayName",
            "brandName",
            "articleTypeName",
            "baseColour",
            "imageURL"
        ]
    ].head()
)

Dataset shape: (18632, 17)


,id,productDisplayName,brandName,articleTypeName,baseColour,imageURL
0,57896,Femella Women Off White Shorts,Femella,Shorts,Off White,http://assets.myntassets.com/v1/images/style/p...
1,51041,Enamor Women Ivory Coloured Briefs,Enamor,Briefs,White,http://assets.myntassets.com/v1/images/style/p...
2,59471,Lucera Silver Ring,Lucera,Ring,Silver,http://assets.myntassets.com/v1/images/style/p...
3,42397,Flying Machine Women Black T-shirt,Flying Machine,Tshirts,Black,http://assets.myntassets.com/v1/images/style/p...
4,38162,Miami Blues Women Sunglasses,Miami Blues,Sunglasses,Brown,http://assets.myntassets.com/v1/images/style/p...


In [10]:
# Load the saved EfficientNetB0 model

best_model = tf.keras.models.load_model("efficientnetb0_best.keras")

print("EfficientNetB0 model loaded successfully.")

EfficientNetB0 model loaded successfully.


<!-- Image Embedding Extraction -->

In [7]:
# Load the selected category dataset
df_recommend = pd.read_csv("fashion_products_selected_v2.csv")

print("Selected category dataset:", df_recommend.shape)
print("Category:", df_recommend["modeling_category"].unique())

Selected category dataset: (18632, 17)
Category: ['Women']


In [8]:
# Check available images
image_dir = "images"

df_recommend["image_path"] = df_recommend["filename"].apply(
    lambda x: os.path.join(image_dir, x)
)

df_recommend["image_exists"] = df_recommend["image_path"].apply(
    os.path.exists
)

print("Total Women products:", len(df_recommend))
print("Images available:", df_recommend["image_exists"].sum())
print("Images missing:", (~df_recommend["image_exists"]).sum())

Total Women products: 18632
Images available: 18630
Images missing: 2


In [9]:
# Keep products with valid image files
df_recommend = df_recommend[
    df_recommend["image_exists"]
].copy()

print("Products available for embedding:", len(df_recommend))

Products available for embedding: 18630


In [14]:
df_recommend["filename"].str.lower().str.rsplit(".", n=1).str[-1].value_counts()

filename
jpg    18630
Name: count, dtype: int64

In [19]:
# Create the embedding dataset
def load_image_for_embedding(image_path):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    )

    image = preprocess_input(image)

    return image

JPEG file. .jpg/.jpeg,
   ↓
Raw image pixels
   ↓
TensorFlow tensor
   ↓
224 × 224 × 3

The channels=3 means RGB.

In [ ]:
# embedding dataset
embedding_ds = tf.data.Dataset.from_tensor_slices(
    df_recommend["image_path"].values
)

embedding_ds = embedding_ds.map(
    load_image_for_embedding,
    num_parallel_calls=AUTOTUNE
)

embedding_ds = embedding_ds.batch(BATCH_SIZE)
embedding_ds = embedding_ds.prefetch(AUTOTUNE)

print("Embedding dataset created successfully.")


Embedding dataset created successfully.


In [22]:
# Generate embeddings ---->tc
import time
start_time = time.time()

embeddings = feature_extractor.predict(
    embedding_ds,
    verbose=1
)

embedding_time = time.time() - start_time

print("Embedding extraction completed.")
print("Embedding shape:", embeddings.shape)
print(
    "Extraction time:",
    round(embedding_time / 60, 2),
    "minutes"
)

583/583 ━━━━━━━━━━━━━━━━━━━━ 2269s 4s/step
Embedding extraction completed.
Embedding shape: (18630, 1280)
Extraction time: 37.83 minutes


18,630 Women products
Each image was passed through our trained EfficientNetB0 feature extractor
Each image is now represented by 1,280 visual features
18,630 images
↓
EfficientNetB0 feature extractor
↓
18,630 × 1,280
↓
Visual embeddings

Each of the 18,630 product images is now represented by a fixed-length vector containing 1,280 visual features.

In [23]:
# Normalize embeddings
from sklearn.preprocessing import normalize

embeddings = normalize(
    embeddings,
    norm="l2"
)

print("Embeddings normalized successfully.")
print("Embedding shape:", embeddings.shape)

Embeddings normalized successfully.
Embedding shape: (18630, 1280)


In [25]:
# Save the embedding
import pickle

embedding_data = {
    "embeddings": embeddings,
    "metadata": df_recommend
}

with open(
    "women_product_embeddings_v2.pkl",
    "wb"
) as f:
    pickle.dump(
        embedding_data,
        f
    )

print("Pickle file saved successfully.")

Pickle file saved successfully.


In [27]:
# verify
with open(
    "women_product_embeddings_v2.pkl",
    "rb"
) as f:
    saved_data = pickle.load(f)

print(
    "Embedding shape:",
    saved_data["embeddings"].shape
)

print(
    "Metadata shape:",
    saved_data["metadata"].shape
)

Embedding shape: (18630, 1280)
Metadata shape: (18630, 19)


women_product_embeddings.pkl
↓
18,630 product embeddings
↓
1,280 features per image

<!-- Image Embedding Extraction -->

- The trained EfficientNetB0 model was successfully reused as a feature extractor by excluding the final 54-class classification layer.

- A total of 18,630 valid Women fashion product images were processed for visual feature extraction.

- Each product image was converted into a fixed-length 1,280-dimensional feature vector using the GlobalAveragePooling2D layer of EfficientNetB0.

- The 1,280-dimensional embeddings represent learned visual characteristics of the products, such as shape, texture, colour patterns, and other image-level features.

- The embeddings provide a numerical representation that allows visually similar products to be compared mathematically.

- The embeddings and corresponding product metadata were stored together in `women_product_embeddings.pkl`, ensuring that each embedding remains aligned with its product information.

- These visual embeddings will be used in the next step to calculate cosine similarity and identify the most visually similar Women fashion products.

<!-- Conclusion -->

The image embedding stage successfully transformed 18,630 product images into structured 1,280-dimensional visual representations. These embeddings form the foundation of the similarity-based fashion recommendation system.

<!-- Similarity Computation -->
